# TrackMate feature-filter explorer

60枚目のTrackMate候補に対して各特徴量の下限・上限を入力し、積集合を元画像へ重ねて確認します。

1. 最初から順にセルを実行します。
2. GUIの数値を変更して **Apply filters** を押します。
3. GUIが表示されない場合は、末尾の辞書入力版を使います。

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root was not found.')

IMAGE_PATH = PROJECT_ROOT / 'outputs' / '042926_MAY08R_FOS_1_retake_c_uint16_scale10000_angle_smoothed' / '042926_MAY08R_FOS_1_retake_c_060.tif'
SPOTS_PATH = PROJECT_ROOT / 'outputs' / 'trackmate_angle_smoothed_test' / '042926_MAY08R_FOS_1_retake_c_060_angle_smoothed_r3_th100_median_spots.pkl'
SAVE_DIR = PROJECT_ROOT / 'outputs' / 'trackmate_filter_explorer'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

image = tiff.imread(str(IMAGE_PATH))
spots = pd.read_pickle(str(SPOTS_PATH)).copy()
spots['CV_INTENSITY_CH1'] = spots['STD_INTENSITY_CH1'] / spots['MEAN_INTENSITY_CH1']
print('Image:', image.shape, image.dtype)
print('Spots:', len(spots))

Image: (4096, 2160) uint16
Spots: 27811


In [6]:
FEATURES = [
    ('QUALITY', 'Q'),
    ('MIN_INTENSITY_CH1', 'Min'),
    ('MEAN_INTENSITY_CH1', 'Mean'),
    ('STD_INTENSITY_CH1', 'SD'),
    ('CV_INTENSITY_CH1', 'CV'),
    ('CONTRAST_CH1', 'Contrast'),
    ('SNR_CH1', 'SNR'),
    ('MAX_INTENSITY_CH1', 'Max'),
]

DEFAULT_BOUNDS = {
    'QUALITY': (100.0, None),
    'MIN_INTENSITY_CH1': (100.0, 1000.0),
    'MEAN_INTENSITY_CH1': (800.0, 5000.0),
    'STD_INTENSITY_CH1': (None, None),
    'CV_INTENSITY_CH1': (0.25, 1.2),
    'CONTRAST_CH1': (0.13, None),
    'SNR_CH1': (0.44, None),
    'MAX_INTENSITY_CH1': (None, None),
}

def normalized_bounds(bounds):
    result = {}
    for column, _ in FEATURES:
        observed_min = float(spots[column].min())
        observed_max = float(spots[column].max())
        low, high = bounds.get(column, (None, None))
        result[column] = (
            -np.inf if low is None else float(low),
            np.inf if high is None else float(high),
        )
    return result

def filter_spots(bounds):
    bounds = normalized_bounds(bounds)
    mask = pd.Series(True, index=spots.index)
    for column, _ in FEATURES:
        low, high = bounds[column]
        mask &= (spots[column] > low) & (spots[column] < high)
    return spots.loc[mask].copy(), bounds

def show_filtered(bounds, point_size=10.0, gamma=0.7, vmax=5500.0, figure_scale=0.65, save=False, suffix='selection'):
    selected, used_bounds = filter_spots(bounds)
    fig, ax = plt.subplots(figsize=(8 * figure_scale, 14 * figure_scale), dpi=120, facecolor='black')
    ax.imshow(image, cmap='gray', norm=PowerNorm(gamma=gamma, vmin=0, vmax=vmax))
    ax.scatter(
        selected['POSITION_X'], selected['POSITION_Y'], s=point_size,
        facecolors='none', edgecolors='#ff3030', linewidths=0.45
    )
    ax.set_title('Selected {:,} / {:,} spots ({:.1f}%)'.format(
        len(selected), len(spots), 100.0 * len(selected) / len(spots)
    ), color='white')
    ax.axis('off')
    fig.tight_layout(pad=0.15)
    if save:
        png_path = SAVE_DIR / '{}.png'.format(suffix)
        pkl_path = SAVE_DIR / '{}_spots.pkl'.format(suffix)
        csv_path = SAVE_DIR / '{}_spots.csv'.format(suffix)
        fig.savefig(str(png_path), facecolor='black', bbox_inches='tight')
        selected.to_pickle(str(pkl_path))
        selected.to_csv(str(csv_path), index=False)
        print('Saved:', png_path)
        print('Saved:', pkl_path)
    plt.show()
    summary = pd.DataFrame([
        {'feature': label, 'lower_exclusive': used_bounds[column][0], 'upper_exclusive': used_bounds[column][1]}
        for column, label in FEATURES
    ])
    return selected, summary

## Interactive GUI

各行のチェックを外すと、その特徴量はフィルタから除外されます。上下限はどちらも排他的です。

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output

controls = {}
rows = [widgets.HTML('<b style="display:inline-block;width:95px">Feature</b><b style="display:inline-block;width:135px">Lower</b><b style="display:inline-block;width:135px">Upper</b><b>Use</b>')]
for column, label in FEATURES:
    observed_min = float(spots[column].min())
    observed_max = float(spots[column].max())
    requested_low, requested_high = DEFAULT_BOUNDS[column]
    margin = max(1e-9, (observed_max - observed_min) * 1e-6)
    default_low = observed_min - margin if requested_low is None else requested_low
    default_high = observed_max + margin if requested_high is None else requested_high
    low_widget = widgets.FloatText(value=default_low, layout=widgets.Layout(width='135px'))
    high_widget = widgets.FloatText(value=default_high, layout=widgets.Layout(width='135px'))
    use_widget = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='45px'))
    controls[column] = (low_widget, high_widget, use_widget, observed_min, observed_max)
    rows.append(widgets.HBox([
        widgets.HTML('<span style="display:inline-block;width:95px">{}</span>'.format(label)),
        low_widget, high_widget, use_widget,
    ]))

point_size_widget = widgets.FloatText(value=10.0, description='Point size:', layout=widgets.Layout(width='210px'))
gamma_widget = widgets.FloatText(value=0.7, description='Gamma:', layout=widgets.Layout(width='190px'))
vmax_widget = widgets.FloatText(value=5500.0, description='Display max:', layout=widgets.Layout(width='230px'))
figure_scale_widget = widgets.FloatText(value=0.65, description='Figure scale:', layout=widgets.Layout(width='210px'))
save_widget = widgets.Checkbox(value=False, description='Save PNG/PKL/CSV')
suffix_widget = widgets.Text(value='selection', description='File suffix:', layout=widgets.Layout(width='300px'))
apply_button = widgets.Button(description='Apply filters', button_style='primary', icon='filter')
output = widgets.Output()

def current_widget_bounds():
    bounds = {}
    for column, _ in FEATURES:
        low_widget, high_widget, use_widget, observed_min, observed_max = controls[column]
        if use_widget.value:
            bounds[column] = (low_widget.value, high_widget.value)
        else:
            bounds[column] = (None, None)
    return bounds

def apply_filters(_=None):
    with output:
        clear_output(wait=True)
        selected, summary = show_filtered(
            current_widget_bounds(),
            point_size=point_size_widget.value, gamma=gamma_widget.value,
            vmax=vmax_widget.value, figure_scale=max(0.1, figure_scale_widget.value),
            save=save_widget.value,
            suffix=suffix_widget.value.strip() or 'selection',
        )
        display(summary)

apply_button.on_click(apply_filters)
display(widgets.VBox(rows + [
    widgets.HBox([point_size_widget, gamma_widget, vmax_widget, figure_scale_widget]),
    widgets.HBox([save_widget, suffix_widget, apply_button]),
    output,
]))
apply_filters()

## GUIを使わない場合

辞書を書き換えてセルを再実行します。`None`は観測最小値または最大値を使用します。

In [ ]:
MANUAL_BOUNDS = {
    'QUALITY': (100, None),
    'MIN_INTENSITY_CH1': (100, 1000),
    'MEAN_INTENSITY_CH1': (800, 5000),
    'STD_INTENSITY_CH1': (None, None),
    'CV_INTENSITY_CH1': (0.25, 1.2),
    'CONTRAST_CH1': (0.13, None),
    'SNR_CH1': (0.44, None),
    'MAX_INTENSITY_CH1': (None, None),
}
manual_selected, manual_summary = show_filtered(
    MANUAL_BOUNDS, point_size=10, gamma=0.7, vmax=5500, figure_scale=0.65,
    save=False, suffix='manual_selection',
)
display(manual_summary)